# HalDetect Phase 10: Full DeBERTa-v3-Large Rescore
Upload `nli_full_rescore_worklist.json` to Kaggle, ensure a T4 GPU is attached, and run all.

In [ ]:
!pip install transformers accelerate sentencepiece torch

In [ ]:
import json
import time
import torch
from transformers import pipeline

# CHANGE THIS TO MATCH KAGGLE'S UPLOAD PATH
INPUT_FILE = "/kaggle/input/YOUR_DATASET_NAME_HERE/nli_full_rescore_worklist.json"
OUTPUT_FILE = "nli_full_rescore_scores.json"

print(f"Loading {INPUT_FILE}...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

items = data["items"]
print(f"Loaded {len(items)} claims to evaluate.")

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print(f"Loading cross-encoder/nli-deberta-v3-large on device={device}...")
classifier = pipeline("text-classification", 
                      model="cross-encoder/nli-deberta-v3-large", 
                      device=device,
                      top_k=None) # top_k=None returns all scores
print("Model loaded successfully!")

In [ ]:
results = []
t0 = time.time()

for i, row in enumerate(items, 1):
    claim = row["claim"]
    passage = row["passage"]
    cid = row["claim_id"]
    
    if not passage.strip() or passage.startswith("NLI "):
        # Clean up the passage string if it has the NLI prefix from the old cache
        if "|" in passage:
            passage = passage.split("|", 1)[1].strip().strip("'")
        
    if not passage.strip():
        results.append({
            "claim_id": cid,
            "entail": 0.0,
            "contradict": 0.0,
            "verdict": "UNVERIFIABLE",
            "verdict_confidence": 0.0
        })
        continue
        
    try:
        res = classifier({"text": passage, "text_pair": claim})
        
        # DeBERTa-v3 outputs CONTRADICTION, ENTAILMENT, NEUTRAL
        scores = {item['label'].upper(): item['score'] for item in res}
        
        entail = scores.get("ENTAILMENT", 0.0)
        contra = scores.get("CONTRADICTION", 0.0)
        
        if entail > contra:
            verdict = "TRUE"
            conf = entail
        else:
            verdict = "FALSE"
            conf = contra
            
        results.append({
            "claim_id": cid,
            "entail": round(entail, 4),
            "contradict": round(contra, 4),
            "verdict": verdict,
            "verdict_confidence": round(conf, 4)
        })
    except Exception as e:
        print(f"Error on {cid}: {e}")
        
    if i % 50 == 0 or i == len(items):
        print(f"Processed {i}/{len(items)} ({(time.time()-t0):.1f}s)")

print("Finished inference!")
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"Saved results to {OUTPUT_FILE}")